# LCEL
- LangChain Expression Language (LCEL)은 LangChain 라이브러리에서 제공하는 선언적 방식의 인터페이스로, 
  
  복잡한 LLM (Large Language Model) 애플리케이션을 구축하고 실행하기 위한 도구

* LCEL은 LLM, 프롬프트, 검색기, 메모리 등 다양한 컴포넌트를 조합하여 강력하고 유연한 AI 시스템을 구축 가능

<br>

#### 주요 특징
- **선언적 구문**: 복잡한 로직을 간결하고 읽기 쉬운 방식으로 표현
- **모듈성**: 다양한 컴포넌트를 쉽게 조합하고 재사용
- **유연성**: 다양한 유형의 LLM 애플리케이션을 구축
- **확장성**: 사용자 정의 컴포넌트를 쉽게 통합
- **최적화**: 실행 시 자동으로 최적화를 수행

<br>

<hr>

<br>

## RunnablePassthrough
- `invoke()` 메서드를 통해 입력된 데이터를 그대로 반환
-  다음과 같은 시나리오에서 유용
   - 데이터를 변환하거나 수정할 필요가 없는 경우
   - 파이프라인의 특정 단계를 건너뛰어야 하는 경우
   - 디버깅 또는 테스트 목적으로 데이터 흐름을 모니터링해야 하는 경우

<br>

- **Runnable = 입력 $\rightarrow$ 출력 변환을 수행하는, 체이닝 가능한 추상 연산자**
  - 어떤 입력(Input)을 받아서, 어떤 출력(Output)을 반환할 수 있는 실행 가능한 단위
  - LCEL 전체는 Runnable 들을 조합해 만든 실행 그래프

| LCEL 객체             | Runnable로서의 역할             |
| ------------------- | -------------------------- |
| PromptTemplate      | 입력 dict → 포맷된 문자열 변환       |
| ChatModel           | 문자열 → LLM 응답 반환            |
| Retriever           | query → relevant documents |
| OutputParser        | LLM 결과 → Python 객체로 변환     |
| RunnableLambda      | 아무 Python 함수도 Runnable화    |
| RunnableParallel    | 병렬 구동 Runnable             |
| RunnableMap         | 여러 key별 runnable 실행        |
| RunnablePassthrough | 입력을 그대로 전달                 |



<br>

### 데이터 전달
- `RunnablePassthrough` 는 입력을 변경하지 않고 그대로 전달하거나 추가 키를 더하여 전달
  - `assign`과 함께 호출된 `RunnablePassthrough(RunnablePassthrough.assign(...))`는 입력을 받아 `assign` 함수에 전달된 추가 인자를 더함
- `RunnableParallel` 클래스를 사용하여 병렬로 실행 가능한 작업을 정의
<br>



In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

In [3]:
r = RunnablePassthrough.assign(mult=lambda x: x["num"] * 3)
r.invoke({"num": 1})

{'num': 1, 'mult': 3}

<br>

#### 검색기 예제

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [5]:
vectorstore = FAISS.from_texts(
    ["Teddy is an AI engineer who loves programming!"],
    embedding=OpenAIEmbeddings(),
)

retriever = vectorstore.as_retriever()

In [6]:
template = """Answer the question based only on the following context:
{context}  

Question: {question}"""

prompt = ChatPromptTemplate.from_template(
    template
)

In [7]:
model = ChatOpenAI(model_name="gpt-4o-mini")

In [8]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [9]:
chain.invoke("테디의 직업은 무엇입니까?")

'테디의 직업은 AI 엔지니어입니다.'

<br>

<hr>

<br>

## Runnable 구조(그래프) 검토

<br>

### 그래프 구성 확인

- 체인의 그래프에서 노드

In [10]:
chain.get_graph().nodes

{'10fc09f3c1b54b17a6fa1251587fbc05': Node(id='10fc09f3c1b54b17a6fa1251587fbc05', name='Parallel<context,question>Input', data=<class 'langchain_core.runnables.base.RunnableParallel<context,question>Input'>, metadata=None),
 '87f7d1e66321466695ac724e3e90ec97': Node(id='87f7d1e66321466695ac724e3e90ec97', name='Parallel<context,question>Output', data=<class 'langchain_core.utils.pydantic.RunnableParallel<context,question>Output'>, metadata=None),
 'f07ffe13cf79494c93c1a4c4de228c0d': Node(id='f07ffe13cf79494c93c1a4c4de228c0d', name='VectorStoreRetriever', data=VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000013ACFE02C90>, search_kwargs={}), metadata=None),
 '6dbde686c3b1431e93d0c2709314b203': Node(id='6dbde686c3b1431e93d0c2709314b203', name='Passthrough', data=RunnablePassthrough(), metadata=None),
 '7145fc81c46142268511b10cb14b7bc6': Node(id='7145fc81c46142268511b10cb14b7bc6', name='ChatPromptTemplate', dat

- 체인의 그래프에서 엣지

In [11]:
chain.get_graph().edges

[Edge(source='c54d979be3b24b16bc5bdaadaaca5cbe', target='918e0023737f4e569efc385d536e1428', data=None, conditional=False),
 Edge(source='918e0023737f4e569efc385d536e1428', target='1d149340bc7f439e9dcd8dd4fa41e28a', data=None, conditional=False),
 Edge(source='c54d979be3b24b16bc5bdaadaaca5cbe', target='9031c38d239147e28d57805cbac63a95', data=None, conditional=False),
 Edge(source='9031c38d239147e28d57805cbac63a95', target='1d149340bc7f439e9dcd8dd4fa41e28a', data=None, conditional=False),
 Edge(source='1d149340bc7f439e9dcd8dd4fa41e28a', target='802def60232b431e8d05753b7624a753', data=None, conditional=False),
 Edge(source='802def60232b431e8d05753b7624a753', target='f4b52030a79e4001b1c394324eb79907', data=None, conditional=False),
 Edge(source='6b4fb80e00f049b5b7d7dd646a10ec7e', target='5f54ce47007b42f288548403ae85c781', data=None, conditional=False),
 Edge(source='f4b52030a79e4001b1c394324eb79907', target='6b4fb80e00f049b5b7d7dd646a10ec7e', data=None, conditional=False)]

- 그래프 출력

In [12]:
chain.get_graph().print_ascii()

           +---------------------------------+         
           | Parallel<context,question>Input |         
           +---------------------------------+         
                    **               **                
                 ***                   ***             
               **                         **           
+----------------------+              +-------------+  
| VectorStoreRetriever |              | Passthrough |  
+----------------------+              +-------------+  
                    **               **                
                      ***         ***                  
                         **     **                     
           +----------------------------------+        
           | Parallel<context,question>Output |        
           +----------------------------------+        
                             *                         
                             *                         
                             *                  

- 프롬프트 출력

In [13]:
print(chain.get_prompts())

[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}  \n\nQuestion: {question}'), additional_kwargs={})])]


<br>

<hr>

<br>

## 